# AQI Jakarta — sapuan horizon × seed (protokol bebas-leakage)

Menjawab sisa RQ1: **apakah keunggulan model bertahan saat horizon diperpanjang 5 → 24 jam, dan apakah stabil antar-seed?**

Yang penting dipahami soal "horizon" di kode kita: horizon efektif = `window (5) + --horizon - 1`. Jadi untuk menguji 5/7/12/24 jam, `--horizon` yang dipakai 1/3/8/20. Nilai yang dilaporkan di tabel selalu **jam**, bukan nilai `--horizon`.

Semua run memakai protokol bersih `--leak-safe` dan lengan **base** (tanpa cuaca) — lengan cuaca sudah terbukti kalah di jendela penuh (0,6595 vs 0,6035), jadi tidak diulang di sini.

**Setelan sesi (sidebar kanan → Session options):**
- Accelerator: **GPU T4 x2** (kode memaksa 1 GPU)
- Internet: **On**

**Biar tidak kehabisan sesi.** Satu run ≈ **50 menit** di T4 (10 IMF × 100 epoch, terukur dari run 11 Okt). Default = 4 horizon × 2 seed = **8 run ≈ 7 jam**; batas sesi Kaggle 12 jam. Kalau waktu mepet, kecilkan `SEEDS` jadi `[0]` (4 run ≈ 3,5 jam) dan jalankan sisa seed di sesi berikutnya — sel "seed 2" di bawah disiapkan untuk itu. Sel loop akan **melewati** run yang CSV-nya sudah ada, jadi mengulang notebook tidak membuang kuota.

Hasil tiap run ikut ke output `/kaggle/working` setelah **Save Version**.

**Ambil repo + data.**

In [1]:
import os

REPO_URL = "https://github.com/madajabbar/riset-aqi-jakarta_ML.git"
BRANCH   = "main"
REPO     = "/kaggle/working/riset-aqi-jakarta_ML"

if not os.path.exists(REPO):
    !git clone -b {BRANCH} {REPO_URL} {REPO}
else:
    !git -C {REPO} fetch origin {BRANCH} && git -C {REPO} reset --hard origin/{BRANCH}

os.chdir(REPO)
os.makedirs("our_model/results", exist_ok=True)
START, END = "2022-08-15", "2026-10-01"
!git -C {REPO} log --oneline -1
print(os.getcwd())

Cloning into '/kaggle/working/riset-aqi-jakarta_ML'...
remote: Enumerating objects: 165, done.
remote: Counting objects: 100% (165/165), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 165 (delta 88), reused 119 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (165/165), 3.34 MiB | 24.41 MiB/s, done.
Resolving deltas: 100% (88/88), done.
84029be (HEAD -> main, origin/main, origin/HEAD) feat(sweep): nama checkpoint memuat --seed dan --horizon + notebook sapuan horizon x seed
/kaggle/working/riset-aqi-jakarta_ML


**Pasang `EMD-signal`.** Kaggle sudah punya torch/sklearn/pandas/matplotlib.

In [2]:
%pip install EMD-signal

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 3.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


import os, torch
assert os.path.exists("our_data/jakarta_aqi.csv"), "data hilang — cek git clone"
from PyEMD import EMD
print("PyEMD OK | torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.device_count())
src = open("our_model/train.py").read()
assert "ow{a.horizon}" in src and "a.seed" in src, "kode lama: penanda ow/seed belum ada di run_id"
print("train.py memuat penanda checkpoint ow+seed: OK")

In [3]:
import os, torch
assert os.path.exists("our_data/jakarta_aqi.csv"), "data hilang — cek git clone"
from PyEMD import EMD
print("PyEMD OK | torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.device_count())
# penanda wajib: run_id harus memuat ow.. dan s.. supaya checkpoint antar-run tidak bertabrakan
!grep -n "ow{a.horizon}\|a.seed" our_model/train.py | head -3

PyEMD OK | torch 2.11.0+cu128 | CUDA: True | GPU: 2
85:    set_seed(a.seed)
182:              + (f"-ow{a.horizon}" if a.horizon != 1 else "")
183:              + (f"-s{a.seed}" if a.seed else ""))


## Sapuan utama — 4 horizon × 2 seed

Loop ini idempoten: run yang CSV-nya sudah ada akan dilewati, jadi aman diulang setelah sesi mati.
Perhatikan di log tiap run: baris `EMD leak-safe: N kanal sejajar …` (N sama untuk ketiga blok) dan baris akhir `RMSE … | MAE … | MAPE …%`.

In [4]:
import os
HORIZONS = [5, 7, 12, 24]   # JAM efektif
SEEDS    = [0, 1]

for h in HORIZONS:
    ow = h - 4              # horizon efektif = window(5) + ow - 1
    for s in SEEDS:
        ow_sfx = "" if ow == 1 else f"-ow{ow}"
        s_sfx  = "" if s == 0 else f"-s{s}"
        csv = f"our_model/results/jakarta-us_aqi-base-{START}_{END}-exoimf-leaksafe{ow_sfx}{s_sfx}-win5-h{ow}.csv"
        if os.path.exists(csv):
            print(f"SKIP {h} jam seed {s}: sudah ada"); continue
        print(f"\n===== horizon {h} jam (--horizon {ow}) seed {s} =====", flush=True)
        !PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES=0 python our_model/train.py --leak-safe --horizon {ow} --seed {s} --start {START} --end {END} 2>&1 | tee our_model/results/sweep-h{h}-s{s}.log


===== horizon 5 jam (--horizon 1) seed 0 =====
split: train 25285 | val 5419 | test 5418 windows
EMD leak-safe: 10 kanal sejajar dari train 13 mode | val 10 mode | test 10 mode in 18s
/kaggle/working/riset-aqi-jakarta_ML/our_model/model.py:39: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=num_layers)
Training IMF 1/10 (target = IMF 1) ...
IMF1 ep20 loss 0.00003 3s
IMF1 ep40 loss 0.00002 3s
IMF1 ep60 loss 0.00001 3s
IMF1 ep80 loss 0.00001 3s
IMF1 ep100 loss 0.00001 3s
IMF1: val transformer 0.0208 | linear 0.0345 -> tf
[CHECKPOINT SAVED] IMF 1 state saved to disk.
/kaggle/working/riset-aqi-jakarta_ML/our_model/model.py:39: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_fi

## Opsional — seed ke-3 (hanya kalau sesi masih punya waktu)

Sesuai plan: 3 seed supaya bisa dilaporkan rata-rata ± simpangan baku. Kalau 8 run di atas sudah memakan sesi, jalankan sel ini di sesi terpisah (butuh ±3,5 jam).

In [5]:
import os
HORIZONS = [5, 7, 12, 24]
SEEDS    = [2]

for h in HORIZONS:
    ow = h - 4
    for s in SEEDS:
        ow_sfx = "" if ow == 1 else f"-ow{ow}"
        csv = f"our_model/results/jakarta-us_aqi-base-{START}_{END}-exoimf-leaksafe{ow_sfx}-s{s}-win5-h{ow}.csv"
        if os.path.exists(csv):
            print(f"SKIP {h} jam seed {s}: sudah ada"); continue
        print(f"\n===== horizon {h} jam (--horizon {ow}) seed {s} =====", flush=True)
        !PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES=0 python our_model/train.py --leak-safe --horizon {ow} --seed {s} --start {START} --end {END} 2>&1 | tee our_model/results/sweep-h{h}-s{s}.log


===== horizon 5 jam (--horizon 1) seed 2 =====
split: train 25285 | val 5419 | test 5418 windows
EMD leak-safe: 10 kanal sejajar dari train 13 mode | val 10 mode | test 10 mode in 17s
/kaggle/working/riset-aqi-jakarta_ML/our_model/model.py:39: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=num_layers)
Training IMF 1/10 (target = IMF 1) ...
IMF1 ep20 loss 0.00003 3s
IMF1 ep40 loss 0.00002 3s
IMF1 ep60 loss 0.00001 2s
IMF1 ep80 loss 0.00001 3s
IMF1 ep100 loss 0.00001 3s
IMF1: val transformer 0.0210 | linear 0.0345 -> tf
[CHECKPOINT SAVED] IMF 1 state saved to disk.
/kaggle/working/riset-aqi-jakarta_ML/our_model/model.py:39: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_fi

## Tabel hasil: rata-rata ± simpangan baku antar-seed

Baseline pembanding yang wajib ada di paper: **persistensi** — tebakan "nilai h jam ke depan sama dengan nilai sekarang". Kolom `kali_lebih_baik` = RMSE persistensi ÷ RMSE model. Kalau angka ini mendekati 1, model tidak berguna di horizon itu.

In [6]:
import glob, re, numpy as np, pandas as pd

aq = (pd.read_csv("our_data/jakarta_aqi.csv", parse_dates=["datetime"])
        .set_index("datetime")["us_aqi"]).sort_index()

def met(pred, truth):
    e = pred - truth
    return (np.sqrt((e ** 2).mean()), np.abs(e).mean(), (np.abs(e) / np.abs(truth)).mean() * 100)

files = sorted(set(glob.glob("our_model/results/jakarta-us_aqi-base-*-leaksafe-*win5-h*.csv")))
rows = []
for f in files:
    m_ow = re.search(r"-ow(\d+)-", f)
    ow = int(m_ow.group(1)) if m_ow else 1
    m_s = re.search(r"-s(\d+)-win5", f)
    s = int(m_s.group(1)) if m_s else 0
    d = pd.read_csv(f, index_col=0, parse_dates=True)
    h = 5 + ow - 1
    r, m, mp = met(d["pred"].values, d["truth"].values)
    p = aq.reindex(d.index - pd.Timedelta(hours=h)).values
    ok = ~np.isnan(p)
    rp = met(p[ok], d["truth"].values[ok])[0]
    rows.append({"h_jam": h, "ow": ow, "seed": s, "n_titik": len(d),
                 "RMSE": round(r, 4), "MAE": round(m, 4), "MAPE_%": round(mp, 2),
                 "RMSE_persistensi": round(rp, 4), "kali_lebih_baik": round(rp / r, 2)})

t = pd.DataFrame(rows).sort_values(["h_jam", "seed"])
print(t.to_string(index=False))
print("\nrata-rata ± std antar-seed:")
print(t.groupby("h_jam")[["RMSE", "MAE", "MAPE_%", "kali_lebih_baik"]]
        .agg(["mean", "std"]).round(4).to_string())
t.to_csv("our_model/results/tabel_horizon_seeds.csv", index=False)
print("\ntersimpan: our_model/results/tabel_horizon_seeds.csv")

 h_jam  ow  seed  n_titik    RMSE    MAE  MAPE_%  RMSE_persistensi  kali_lebih_baik
     5   1     0     5415  0.6035 0.3641    0.23           11.0805            18.36
     5   1     1     5415  0.5432 0.3307    0.21           11.0805            20.40
     5   1     2     5415  0.5203 0.2918    0.18           11.0805            21.30
     7   3     0     5413  1.7332 0.9411    0.58           12.0493             6.95
     7   3     1     5413  4.8055 2.5510    1.52           12.0493             2.51
     7   3     2     5413  1.7822 0.9547    0.58           12.0493             6.76
    12   8     0     5408  3.6709 2.1961    1.35           14.1141             3.84
    12   8     1     5408  3.0112 1.8252    1.14           14.1141             4.69
    12   8     2     5408  2.7401 1.6234    1.00           14.1141             5.15
    24  20     0     5396  7.9317 4.8686    3.15           17.1268             2.16
    24  20     1     5396  6.9243 4.6403    2.90           17.1268          

## Simpan hasil ke output notebook

In [7]:
import glob, shutil, os
os.makedirs("/kaggle/working/outputs", exist_ok=True)
for f in glob.glob("our_model/results/*.log") + glob.glob("our_model/results/*.csv") + glob.glob("our_model/results/*.png"):
    shutil.copy(f, "/kaggle/working/outputs/")
shutil.make_archive("/kaggle/working/horizon-seeds-results", "zip", "our_model/results")
print(sorted(os.listdir("/kaggle/working/outputs")))

['jakarta-us_aqi-base-2022-08-15_2026-10-01-exoimf-leaksafe-ow20-s1-win5-h20.csv', 'jakarta-us_aqi-base-2022-08-15_2026-10-01-exoimf-leaksafe-ow20-s1-win5-h20.png', 'jakarta-us_aqi-base-2022-08-15_2026-10-01-exoimf-leaksafe-ow20-s2-win5-h20.csv', 'jakarta-us_aqi-base-2022-08-15_2026-10-01-exoimf-leaksafe-ow20-s2-win5-h20.png', 'jakarta-us_aqi-base-2022-08-15_2026-10-01-exoimf-leaksafe-ow20-win5-h20.csv', 'jakarta-us_aqi-base-2022-08-15_2026-10-01-exoimf-leaksafe-ow20-win5-h20.png', 'jakarta-us_aqi-base-2022-08-15_2026-10-01-exoimf-leaksafe-ow3-s1-win5-h3.csv', 'jakarta-us_aqi-base-2022-08-15_2026-10-01-exoimf-leaksafe-ow3-s1-win5-h3.png', 'jakarta-us_aqi-base-2022-08-15_2026-10-01-exoimf-leaksafe-ow3-s2-win5-h3.csv', 'jakarta-us_aqi-base-2022-08-15_2026-10-01-exoimf-leaksafe-ow3-s2-win5-h3.png', 'jakarta-us_aqi-base-2022-08-15_2026-10-01-exoimf-leaksafe-ow3-win5-h3.csv', 'jakarta-us_aqi-base-2022-08-15_2026-10-01-exoimf-leaksafe-ow3-win5-h3.png', 'jakarta-us_aqi-base-2022-08-15_2026-10